# Bengaluru Climate Risk Scanner
## Notebook 04: Mapping the real September 2022 flood (Phase 4)

This notebook builds the **ground truth** that Notebook 05 uses to test the flood model. It uses two independent sources:

1. **Sentinel-1 radar (change detection).** Radar sees through cloud, which matters in the monsoon. Calm water reflects the radar beam away from the satellite, so flooded ground turns much **darker** than it was before the flood. Comparing an August 2022 image with the first image after the 4 September rain shows which cells darkened.
2. **News-reported flooded localities.** Radar is weak in dense cities (buildings bounce the signal back) and Sentinel-1 may have passed days after the peak. A list of places reported flooded in the news is a second, independent check.

**Outputs**

| File | What it holds | Used by |
|---|---|---|
| `data/raw/s1_diff_db.tif`, `s1_post_db.tif`, `jrc_water_occurrence.tif` | Radar change, post-flood backscatter, permanent-water frequency | Re-running detection with new thresholds |
| `flood_2022_observed.tif` | 1 = flooded in Sept 2022 (radar), 0 = not | Validation (Notebook 05) |
| `bengaluru.gpkg` → `flood_2022_observed` | The same, as polygons | GeoServer WMS |
| `bengaluru.gpkg` → `flood_2022_reports` | News-reported flooded localities (points) | Validation and the map |

**Before you start:** run Notebooks 01 and 02. Uses the same Earth Engine login.

## 1. Install libraries

In [ ]:
!pip install -q geemap rasterio

## 2. Connect Google Drive and load the configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, shutil
PROJECT_DIR = '/content/drive/MyDrive/bengaluru-climate-risk'
with open(f'{PROJECT_DIR}/config.json') as f:
    CONFIG = json.load(f)
RAW, OUT, GPKG = CONFIG['paths']['raw'], CONFIG['paths']['processed'], CONFIG['paths']['gpkg']
WORK = '/content/work'
os.makedirs(WORK, exist_ok=True)
print('Flood event windows:', CONFIG['flood_event'])

## 3. Connect to Google Earth Engine
If the sign-in pop-up fails ("credential propagation was unsuccessful"), set `AUTH_MODE = 'notebook'`.

In [ ]:
# [EE]
import ee, geemap
EE_PROJECT = CONFIG['ee_project']
AUTH_MODE = None
if EE_PROJECT == 'YOUR-EE-PROJECT-ID':
    raise ValueError('Set your Earth Engine project ID in config.json (Notebook 01, section 3).')
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode=AUTH_MODE) if AUTH_MODE else ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print('Earth Engine ready:', ee.String('OK').getInfo())

## 4. Detection settings
Change these and rerun from section 8 to tune the detection; the radar files don't need downloading again.

| Setting | Meaning |
|---|---|
| `diff_db` | A cell is a flood candidate if it got at least this much darker (post − pre, in decibels). −3 dB means the returned signal halved. |
| `post_db` | ...and is dark enough to be water after the flood. Open water in VV is usually below −18 dB; −15 dB is a little lenient for shallow, urban water. |
| `max_slope_deg` | Water does not stand on slopes; steeper cells are removed (they are usually radar shadow). |
| `max_hand_m` | Cells far above the nearest drainage are unlikely to flood; removes false alarms on hilltops. |
| `permanent_water_pct` | Cells that were water in more than this % of the 1984–2021 record (JRC Global Surface Water) are permanent lakes, not flood. |
| `min_patch_cells` | Removes isolated specks smaller than this (radar speckle). |

In [ ]:
S1 = {
    "diff_db": -3.0,
    "post_db": -15.0,
    "max_slope_deg": 5.0,
    "max_hand_m": 15.0,
    "permanent_water_pct": 50,
    "min_patch_cells": 5,
    "speckle_radius_m": 50,
}
ev = CONFIG['flood_event']
print('Before-flood window:', ev['pre_start'], 'to', ev['pre_end'])
print('After-flood window: ', ev['post_start'], 'to', ev['post_end'])

## 5. Which Sentinel-1 images exist, and which pairs to compare
Change detection only works between images taken from the **same orbit** (the same viewing angle); otherwise buildings and slopes look different for geometric reasons, not because of water. For each orbit the cell picks the latest image before the flood and the earliest image after it.

Only Sentinel-1A was working in 2022, so each orbit revisits about every 12 days. If no pair is found, widen `post_end` in `config.json` (under `flood_event`) and rerun; the news reports in section 10 still give a validation set.

In [ ]:
# [EE]
import pandas as pd
import geopandas as gpd

aoi_gdf = gpd.read_file(GPKG, layer='aoi')
w, s, e, n = aoi_gdf.total_bounds
m = CONFIG['buffer_deg']
region = ee.Geometry.Rectangle([w - m, s - m, e + m, n + m])

s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(region)
      .filterDate(ev['pre_start'], ee.Date(ev['post_end']).advance(1, 'day'))
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')))

props = ['system:index', 'system:time_start', 'relativeOrbitNumber_start', 'orbitProperties_pass']
rows = s1.reduceColumns(ee.Reducer.toList(len(props)), props).get('list').getInfo()
scenes = pd.DataFrame(rows, columns=['id', 'time', 'orbit', 'pass'])
scenes['date'] = pd.to_datetime(scenes['time'], unit='ms').dt.strftime('%Y-%m-%d')
scenes = scenes.sort_values(['date', 'orbit']).reset_index(drop=True)
print(scenes[['date', 'orbit', 'pass']].drop_duplicates().to_string(index=False))

pairs = []
for orbit, g in scenes.groupby('orbit'):
    pre = g[(g['date'] >= ev['pre_start']) & (g['date'] <= ev['pre_end'])]['date']
    post = g[(g['date'] >= ev['post_start']) & (g['date'] <= ev['post_end'])]['date']
    if len(pre) and len(post):
        pairs.append({'orbit': int(orbit), 'pass': g['pass'].iloc[0], 'pre_date': pre.max(), 'post_date': post.min()})

pairs_df = pd.DataFrame(pairs)
if pairs_df.empty:
    print('\nNo same-orbit pair in the windows. Widen post_end in config.json, or rely on the news reports (section 10).')
else:
    pairs_df['days_after_rain'] = (pd.to_datetime(pairs_df['post_date']) - pd.Timestamp(ev['post_start'])).dt.days
    print('\nPairs used:')
    print(pairs_df.to_string(index=False))

## 6. Radar change detection in Earth Engine
For each pair:
1. **Speckle filter**: radar images are grainy ("speckle"); a 50 m median filter smooths it while keeping edges.
2. Images from the same date are stitched (`mosaic`), separately for before and after.
3. **Difference** = after − before, in dB. Negative = got darker.

With several pairs, the cell keeps the strongest darkening and the darkest after-value at each pixel, so a flood seen by any orbit counts. It also fetches the **JRC Global Surface Water** occurrence layer, to separate permanent lakes from floodwater.

The data in `COPERNICUS/S1_GRD` is already calibrated, terrain-corrected backscatter in dB, so no further calibration is needed.

In [ ]:
# [EE]
def day(img_date):
    d = ee.Date(img_date)
    return d, d.advance(1, 'day')

def smoothed_vv(orbit, date):
    start, end = day(date)
    col = (s1.filter(ee.Filter.eq('relativeOrbitNumber_start', orbit)).filterDate(start, end)
             .select('VV').map(lambda i: i.focal_median(S1['speckle_radius_m'], 'circle', 'meters')))
    return col.mosaic()

diffs, posts = [], []
for p in pairs:
    pre = smoothed_vv(p['orbit'], p['pre_date'])
    post = smoothed_vv(p['orbit'], p['post_date'])
    diffs.append(post.subtract(pre).rename('diff_db'))
    posts.append(post.rename('post_db'))

if pairs:
    diff_img = ee.ImageCollection(diffs).min().rename('diff_db')
    post_img = ee.ImageCollection(posts).min().rename('post_db')
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').unmask(0).rename('occurrence')
print('Change images ready for', len(pairs), 'pair(s)')

## 7. Download the radar layers
Each layer is downloaded on the project's 30 m UTM grid to `data/raw`. Later tuning only re-reads these files.

In [ ]:
# [EE]
import geemap   # imported here too, so this cell still works after a restart that skipped section 3

def export_tif(image, name):
    local = f'{WORK}/{name}.tif'
    geemap.ee_export_image(image.toFloat(), filename=local, scale=CONFIG['resolution_m'],
                           crs=CONFIG['analysis_crs'], region=region, file_per_band=False)
    shutil.copy(local, f'{RAW}/{name}.tif')
    print('Saved', f'{RAW}/{name}.tif')

if pairs:
    export_tif(diff_img, 's1_diff_db')
    export_tif(post_img, 's1_post_db')
export_tif(jrc, 'jrc_water_occurrence')

CONFIG['s1_pairs_used'] = pairs
with open(f'{PROJECT_DIR}/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

## 8. Detect the flood on the project grid
Everything is warped onto the flood model's 30 m grid. A cell counts as **flooded in September 2022** when all of these hold:

1. it got darker by at least `diff_db` **and** is darker than `post_db` after the flood;
2. it is not a permanent lake (OSM lakes, or JRC occurrence above `permanent_water_pct`);
3. it is not steep (slope ≤ `max_slope_deg`) and not high above drainage (HAND ≤ `max_hand_m`);
4. it is inside the city;
5. it belongs to a patch of at least `min_patch_cells` cells.

The slope and HAND come from Notebook 02's `flood_factor_values.tif`.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio import features
from rasterio.warp import reproject, Resampling

with rasterio.open(f'{RAW}/dem_30m.tif') as src:
    profile, transform, crs = src.profile, src.transform, src.crs
    shape, res = (src.height, src.width), src.res[0]
cell_km2 = res * res / 1e6

def on_grid(path, resampling=Resampling.bilinear):
    with rasterio.open(path) as src:
        out = np.full(shape, np.nan, dtype='float32')
        reproject(source=rasterio.band(src, 1), destination=out, src_transform=src.transform, src_crs=src.crs,
                  src_nodata=src.nodata, dst_transform=transform, dst_crs=crs, dst_nodata=np.nan, resampling=resampling)
    return out

have_radar = os.path.exists(f'{RAW}/s1_diff_db.tif')
jrc_occ = on_grid(f'{RAW}/jrc_water_occurrence.tif')
with rasterio.open(f'{OUT}/flood_factor_values.tif') as src:
    hand = src.read(1).astype('float32'); slope = src.read(2).astype('float32')
    hand[hand == src.nodata] = np.nan; slope[slope == src.nodata] = np.nan

boundary = gpd.read_file(GPKG, layer='bbmp_boundary').to_crs(crs)
lakes = gpd.read_file(GPKG, layer='lakes').to_crs(crs)
burn = lambda gdf: features.rasterize(((g, 1) for g in gdf.geometry if g is not None and not g.is_empty),
                                      out_shape=shape, transform=transform, fill=0, dtype='uint8').astype(bool)
city_mask = burn(boundary)
permanent = burn(lakes) | (np.nan_to_num(jrc_occ) > S1['permanent_water_pct'])
land = city_mask & ~permanent

if have_radar:
    diff = on_grid(f'{RAW}/s1_diff_db.tif')
    post = on_grid(f'{RAW}/s1_post_db.tif')
    candidate = (diff <= S1['diff_db']) & (post <= S1['post_db'])
    terrain_ok = (np.nan_to_num(slope, nan=99) <= S1['max_slope_deg']) & (np.nan_to_num(hand, nan=0) <= S1['max_hand_m'])
    flooded = candidate & terrain_ok & land
    # remove specks: drop connected patches smaller than min_patch_cells
    sieved = features.sieve(flooded.astype('uint8'), size=S1['min_patch_cells'], connectivity=8)
    flooded = (sieved == 1) & flooded

    steps = {'Darker and dark enough (all area)': candidate & city_mask,
             '... not permanent water': candidate & land,
             '... flat and low (slope, HAND)': candidate & land & terrain_ok,
             '... after removing specks': flooded}
    for k, v in steps.items():
        print(f'{k:<38} {v.sum() * cell_km2:8.2f} km²')
    print(f'\nObserved flood: {flooded.sum() * cell_km2:.2f} km² '
          f'({100 * flooded.sum() / land.sum():.2f}% of city land)')
else:
    flooded = np.zeros(shape, dtype=bool)
    print('No radar pair was available: the observed-flood layer is empty. Validation will use the news reports.')

## 9. Save the observed flood (raster and polygons)

In [ ]:
from shapely.geometry import shape as to_shape

obs = np.where(city_mask, flooded.astype('uint8'), 255).astype('uint8')
prof = profile.copy()
prof.update(driver='GTiff', dtype='uint8', count=1, nodata=255, compress='deflate', tiled=True, blockxsize=256, blockysize=256)
with rasterio.open(f'{WORK}/flood_2022_observed.tif', 'w', **prof) as dst:
    dst.write(obs, 1)
shutil.copy(f'{WORK}/flood_2022_observed.tif', f'{OUT}/flood_2022_observed.tif')
print('Saved', f'{OUT}/flood_2022_observed.tif')

polys = [to_shape(g) for g, v in features.shapes(flooded.astype('uint8'), mask=flooded, transform=transform, connectivity=8)]
observed_gdf = gpd.GeoDataFrame({'source': ['Sentinel-1'] * len(polys)}, geometry=polys, crs=crs)
observed_gdf['area_m2'] = observed_gdf.area.round(0)
observed_gdf = observed_gdf.to_crs(CONFIG['web_crs'])
print(f'{len(observed_gdf):,} flood polygons, {observed_gdf["area_m2"].sum() / 1e6:.2f} km²')

## 10. News-reported flooded localities
These are the places reported flooded on 4–6 September 2022 (for example in Deccan Herald and Reuters coverage of the Outer Ring Road, Bellandur, Sarjapur Road and Varthur).

**The coordinates below are approximate starting points.** Before using them:
1. Open `MyDrive/bengaluru-climate-risk/data/raw/flood_2022_reports.csv` (created by this cell the first time) in Google Sheets.
2. For each row, check the point on Google Maps, correct `lat`/`lon` to the flooded street or layout named in the article, paste the article link in `source_url`, and set `verified` to `TRUE`.
3. Add more localities you find in news reports (aim for 20 or more). Rerun this cell.

Validation (Notebook 05) reports results for verified points separately, so unverified starting points never inflate the score.

In [ ]:
REPORTS_CSV = f'{RAW}/flood_2022_reports.csv'
STARTER = [
    # name,                                   lat,      lon
    ('Bellandur (Outer Ring Road)',            12.9260, 77.6762),
    ('Marathahalli, Outer Ring Road',          12.9569, 77.7011),
    ('Sarjapur Road (near Wipro, Doddakannelli)', 12.9135, 77.6836),
    ('Varthur',                                12.9406, 77.7470),
    ('Yemalur',                                12.9465, 77.6890),
    ('Epsilon Villas, Yemalur',                12.9480, 77.6920),
    ('Panathur',                               12.9370, 77.7070),
    ('Munnekolala',                            12.9530, 77.7130),
    ('Mahadevapura',                           12.9889, 77.6895),
    ('KR Puram',                               13.0070, 77.6960),
]
if not os.path.exists(REPORTS_CSV):
    pd.DataFrame([{'name': n, 'lat': la, 'lon': lo, 'source_url': '', 'verified': False} for n, la, lo in STARTER]
                 ).to_csv(REPORTS_CSV, index=False)
    print('Created', REPORTS_CSV, '- verify the coordinates and add sources, then rerun.')

reports = pd.read_csv(REPORTS_CSV)
reports['verified'] = reports['verified'].astype(str).str.upper().isin(['TRUE', '1', 'YES'])
reports_gdf = gpd.GeoDataFrame(reports, geometry=gpd.points_from_xy(reports['lon'], reports['lat']), crs='EPSG:4326')
inside_city = reports_gdf.to_crs(crs).within(boundary.geometry.iloc[0])
print(f'{len(reports_gdf)} reported localities, {reports_gdf["verified"].sum()} verified, '
      f'{inside_city.sum()} inside the city boundary')
print(reports_gdf[['name', 'lat', 'lon', 'verified']].to_string(index=False))

## 11. Look at it
Radar-detected flood (dark blue) over the predicted flood classes from Notebook 02, with the reported localities as red dots. You should see the radar flood and the reports mostly sitting in the predicted High (dark) zones, especially in the south-east around Bellandur and Varthur. Notebook 05 puts numbers on this.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

with rasterio.open(f'{OUT}/flood_class.tif') as src:
    pred = src.read(1).astype('float32'); pred[pred == src.nodata] = np.nan; pred[pred == 0] = np.nan
extent = [transform.c, transform.c + shape[1] * res, transform.f - shape[0] * res, transform.f]

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(pred, cmap=ListedColormap(['#f1eef6', '#bdc9e1', '#74a9cf']), norm=BoundaryNorm([0.5, 1.5, 2.5, 3.5], 3),
          extent=extent, interpolation='nearest')
ax.imshow(np.where(flooded, 1, np.nan), cmap=ListedColormap(['#08306b']), extent=extent, interpolation='nearest')
lakes.plot(ax=ax, color='#b7c7d6', zorder=3)
boundary.boundary.plot(ax=ax, color='black', linewidth=0.8, zorder=4)
r = reports_gdf.to_crs(crs)
r.plot(ax=ax, color=np.where(r['verified'], '#d7301f', '#fc9272'), markersize=40, edgecolor='white', linewidth=0.8, zorder=5)
ax.legend(handles=[Patch(color='#f1eef6', label='Predicted Low'), Patch(color='#bdc9e1', label='Predicted Medium'),
                   Patch(color='#74a9cf', label='Predicted High'), Patch(color='#08306b', label='Radar: flooded Sept 2022'),
                   Line2D([], [], marker='o', ls='', color='#d7301f', label='Reported flooded (verified)'),
                   Line2D([], [], marker='o', ls='', color='#fc9272', label='Reported flooded (unverified)')],
          loc='lower left', fontsize=8)
ax.set_title('September 2022 flood: radar and news reports over predicted flood classes')
ax.set_axis_off(); plt.show()

## 12. Save to the GeoPackage and record the settings

In [ ]:
from datetime import datetime, timezone
LOCAL_GPKG = f'{WORK}/bengaluru.gpkg'
shutil.copy(GPKG, LOCAL_GPKG)
if len(observed_gdf):
    observed_gdf.to_file(LOCAL_GPKG, layer='flood_2022_observed', driver='GPKG')
reports_gdf.to_file(LOCAL_GPKG, layer='flood_2022_reports', driver='GPKG')
shutil.copy(LOCAL_GPKG, GPKG)
print('Layers now in the GeoPackage:', gpd.list_layers(GPKG)['name'].tolist())

CONFIG['flood_2022'] = {
    **S1,
    'radar_available': bool(have_radar),
    'observed_km2': round(float(flooded.sum() * cell_km2), 3),
    'reports': int(len(reports_gdf)), 'reports_verified': int(reports_gdf['verified'].sum()),
    'run_at': datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC'),
}
with open(f'{PROJECT_DIR}/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)
print(json.dumps(CONFIG['flood_2022'], indent=2))
print('\nPhase 4 complete. Next: Notebook 05, validation.')